# Tool 4 — Order Lookup: test notebook

Mirrors `test_call_lookup.ipynb` — Tool 4 is the same two-stage design over the
order tables, so the same things are worth pinning:

(a) setup · (b) Stage-1 regex parsing · (c) `_needs_llm` decisions ·
(d) `lookup_orders` end-to-end · (e) merge precedence + LLM-outage fallback
(mocked) · (f) an optional live Stage-2 call.

Run from the repo root's virtualenv. Cells (a)–(e) need **no LLM provider** —
that is the point of Tool 4's lazy client, so they pass with LM Studio down.

In [ ]:
import os, sys, importlib
sys.path.insert(0, os.path.abspath(".."))

import src.tools.order_lookup as ol
from src.config import CONFIG

provider = CONFIG["active_provider"]
print("active provider:", provider)
print("model:", CONFIG["providers"][provider].get("model"))
print("structured-output model:", CONFIG["providers"][provider].get("structured_output_model") or "(same)")
print("orders loaded:", len(ol._load_df()))
print("columns:", list(ol._load_df().columns))

## (b) Stage 1 — regex/enum prefilter, no LLM

`_regex_filters` returns **only** the fields it actually matched. Expected values
are in the trailing comments.

In [ ]:
queries = [
    "where is order 5031",                 # order_id
    "what is the status of ORD-5012",      # order_id
    "#5027",                               # order_id
    "show me Neha's orders",               # customer_name (first name alone)
    "orders for Karan Malhotra",           # customer_name (full name)
    "which orders are out for delivery",   # status (spaced -> out_for_delivery)
    "cancelled audio orders",              # status + category
    "anything shipped by BlueDart",        # status + carrier
    "Neha's last 2 orders",                # customer_name + limit + sort_order
    "first 3 orders",                      # limit + sort_order=asc
    "orders from the last 7 days",         # date range (not a limit)
    "orders from last month",              # (no match) -> Stage 2
]
for q in queries:
    print(f"{q!r:40} -> {ol._regex_filters(q)}")

## (c) `_needs_llm` — when Stage 2 is worth the network call

Escalate only on fuzzy dates, or when Stage 1 matched nothing at all.

In [ ]:
for q in [
    "where is order 5031",           # -> False (order_id resolved)
    "cancelled orders",              # -> False (status resolved)
    "orders from last month",        # -> True  (fuzzy date, nothing matched)
    "recent delivered orders",       # -> True  (fuzzy date wins even with a match)
    "what did I buy",                # -> True  (nothing matched)
    "orders from the last 7 days",   # -> False (date range resolved)
]:
    print(f"{q!r:36} -> needs_llm={ol._needs_llm(q, ol._regex_filters(q))}")

## (d) `lookup_orders` end-to-end (deterministic queries only)

Each order prints as a header line plus a Shipment line and/or a Return line —
present only when that table has a row for the order. `ORD-5027` exercises all
three tables at once (delivered, then a return approved with a refund still
processing); `ORD-5037` exercises an order that has not shipped yet.

In [ ]:
for q in [
    "where is order 5031",
    "ORD-5027",
    "ORD-5037",
    "which orders are out for delivery",
    "show me Neha's orders",
]:
    print("\n" + "=" * 78 + f"\nQUERY: {q}\n" + "=" * 78)
    print(ol.lookup_orders(q))

## (e) Invariants worth an assert (mocked LLM — no network)

Two things must hold no matter what Stage 2 returns:

1. **Regex wins the merge.** A field Stage 1 resolved is never overwritten by the
   LLM; the LLM only fills gaps.
2. **An LLM outage degrades, never raises.** The tool falls back to the
   regex-only filters and still returns a string.

Also pinned: an unfiltered request is capped at `DEFAULT_LIMIT` instead of
dumping the whole table, and a no-match returns the friendly empty string.

In [ ]:
# 1. regex wins the merge — the mock tries to override `carrier` and add dates.
#    NB: the query uses a CARRIER, not a status. A status word sitting next to
#    fuzzy date language is treated as a verb and deliberately left to Stage 2
#    (see _regex_filters), so a status here would be nothing for the LLM to
#    override and the test would prove nothing.
ol._structured_llm.cache_clear()
ol._structured_llm = lambda: type("M", (), {"invoke": staticmethod(
    lambda prompt: ol.OrderFilters(carrier="Delhivery",
                                   date_from="2026-09-01",
                                   date_to="2026-09-30"))})()

merged = ol._resolve_filters("recent BlueDart orders")
assert merged.carrier.value == "BlueDart", f"regex field was overridden! {merged.carrier}"
assert merged.date_from == "2026-09-01", merged.date_from
print("OK: regex `carrier=BlueDart` survived; the LLM only filled the date gap")

# 2. an LLM outage falls back to the regex-only result
ol._structured_llm = lambda: (_ for _ in ()).throw(RuntimeError("simulated LLM outage"))
fallback = ol._resolve_filters("recent BlueDart orders")
assert fallback.carrier.value == "BlueDart"
assert fallback.date_from is None
print("OK: LLM outage degraded to regex-only filters, no exception")

# 3. an unfiltered request is capped, not a full table dump
assert ol.lookup_orders("show me my orders").count("[ORD-") == ol.DEFAULT_LIMIT
print(f"OK: unfiltered request capped at DEFAULT_LIMIT={ol.DEFAULT_LIMIT}")

# 4. no match returns the friendly string, never an exception
assert ol.lookup_orders("order 9999") == "No matching orders found."
print("OK: empty result is a plain-English string")

# 5. an explicit order id wins outright — every other word is the QUESTION, not a
#    filter. ("When will ORD-5033 be delivered?" used to also match
#    status=delivered; ORD-5033 is out_for_delivery, so it returned nothing.)
assert ol._regex_filters("When will ORD-5033 be delivered?") == {"order_id": "ORD-5033"}
assert "ORD-5033" in ol.lookup_orders("When will ORD-5033 be delivered?")
print("OK: an order id suppresses the enum words around it")

# 6. an order id needs its ord/order/# prefix — dates and prices are 4 digits too,
#    and a bogus regex match would ALSO suppress Stage 2.
assert ol._regex_filters("orders after 2026-09-01") == {}   # not ORD-2026
assert ol._regex_filters("my 7999 headphones") == {}        # not ORD-7999
assert ol._needs_llm("orders after 2026-09-01", {}) is True
assert ol._regex_filters("#5012") == {"order_id": "ORD-5012"}
print("OK: bare 4-digit numbers fall through to Stage 2 instead of faking an id")

## (f) Live Stage 2 (slow — needs the active provider running)

`importlib.reload` first, to undo cell (e)'s mocks. This is the only cell that
touches the network; skip it when the provider is down.

In [ ]:
importlib.reload(ol)

for q in ["orders placed last month", "what did Neha buy recently"]:
    print("\n" + "=" * 78 + f"\nQUERY: {q}\n" + "=" * 78)
    print(ol.lookup_orders(q))